# Importing Graph and creating Laplacain Matrix

In [ ]:
import numpy as np
import math

file_path = "P2P-Gnutella.txt"
# Load edges from file (assuming whitespace or comma separated: nodeA nodeB per line)
edges = np.loadtxt(file_path, dtype=int)

# Find the number of nodes (assuming nodes are labeled from 0 to N-1, adjust if necessary)
n_nodes = edges.max() + 1
print(f"n_nodes:{n_nodes}")

# Initialize adjacency matrix
adj = np.zeros((n_nodes, n_nodes), dtype=int)

# Fill adjacency matrix (assuming undirected and unweighted)
for a, b in edges:
    adj[a, b] = 1
    adj[b, a] = 1

In [ ]:
# Degree matrix: diagonal with node degrees
degrees = adj.sum(axis=1)
deg_matrix = np.diag(degrees)

# Laplacian: L = D - A
laplacian = deg_matrix - adj

# Calculating all pairs effective resistance matrix

In [ ]:
def effective_resistance_all_pairs(L):
    # Compute Moore-Penrose pseudoinverse of Laplacian
    L_pinv = np.linalg.pinv(L)

    # Diagonal elements of pseudoinverse
    diag = np.diag(L_pinv)

    # Compute effective resistance matrix R
    # R[i,j] = L^+_ii + L^+_jj - 2 * L^+_ij
    R = diag[:, np.newaxis] + diag[np.newaxis, :] - 2 * L_pinv
    return R

In [ ]:
R = effective_resistance_all_pairs(laplacian)
kirchoff_at_start = R.sum()/2
print(kirchoff_at_start/math.comb(n_nodes,2))
# print(R)

#Number of edges to be added

In [ ]:
5,10,15,20,25

# Running K_median algorithm with one swap

In [ ]:
def forward_greedy_centers(dist_matrix, k):
    n = dist_matrix.shape[0]
    centers = set()
    dist_to_centers = np.full(n, np.inf)  # Track min distance to selected centers

    for _ in range(k):
        best_center = None
        best_cost = np.inf
        for candidate in range(n):
            if candidate in centers:
                continue
            new_dist = np.minimum(dist_to_centers, dist_matrix[:, candidate])
            cost = new_dist.sum()
            if cost < best_cost:
                best_cost = cost
                best_center = candidate
        centers.add(best_center)
        dist_to_centers = np.minimum(dist_to_centers, dist_matrix[:, best_center])
    return centers


def k_median_one_swap_until_optimal(dist_matrix, k):
    n = dist_matrix.shape[0]

    # # Initialize centers randomly
    # centers = set(np.random.choice(n, k, replace=False))
    # Initialize centers using forward greedy
    centers = forward_greedy_centers(dist_matrix, k)
    # print(len(centers))

    def total_cost(centers_set):
        centers_list = list(centers_set)
        dist_to_centers = dist_matrix[:, centers_list]
        min_distances = np.min(dist_to_centers, axis=1)
        # Calculate total cost
        total = np.sum(min_distances)
        return total

    current_cost = total_cost(centers)
    while True:
        improved = False
        for center in list(centers):
            for candidate in range(n):
                if candidate not in centers:
                    new_centers = centers.copy()
                    new_centers.remove(center)
                    new_centers.add(candidate)
                    new_cost = total_cost(new_centers)
                    if new_cost < current_cost:
                        centers = new_centers
                        current_cost = new_cost
                        improved = True
                        break
            if improved:
                break
        if not improved:
            # No improvement found => local optimum
            break
    return list(centers)

#Augmenting edges in 2k+1-median centre's via running greedy on them

In [ ]:
import numpy as np
from tqdm import tqdm

def update_pseudoinverse(L_pinv, u):
    # Sherman-Morrison update for pseudoinverse
    Lp_u = L_pinv @ u
    denom = 1 + u @ Lp_u
    return L_pinv - np.outer(Lp_u, Lp_u) / denom

def update_pseudoinverse_square(L_pinv, L2_pinv, u):
    Lp_u = L_pinv @ u            # Vector
    L2_u = L2_pinv @ u           # Vector
    denom = 1 + u.T @ Lp_u       # Scalar
    uTL2u = u.T @ L2_u           # Scalar

    term1 = np.outer(Lp_u, L2_u) + np.outer(L2_u, Lp_u)   # Symmetric term
    term2 = uTL2u * np.outer(Lp_u, Lp_u)                 # Outer product scaled

    return L2_pinv - term1 / denom + term2 / (denom**2)



def greedy_add_edges_kmedian(L, k,centers_list):
    n = L.shape[0]
    L_current = L.copy()
    L_pinv = np.linalg.pinv(L_current)
    L2_pinv = L_pinv @ L_pinv
    added_edges = []
    for step in range(k):
        adj = -np.minimum(L_current, 0)
        connected = adj > 0
        candidates = [(i, j) for i in centers_list for j in centers_list if not connected[i, j] and i < j]
        # print(candidates)
        if not candidates:
            break
        best_improvement = None
        best_edge = None
        for i, j in tqdm(candidates):
            denom = 1 + (L_pinv[i, i] - 2 * L_pinv[i, j] + L_pinv[j, j])
            num = L2_pinv[i, i] - L2_pinv[i, j] - L2_pinv[j, i] + L2_pinv[j, j]
            improvement = num / denom
            # print(i,j,improvement)
            if best_improvement is None or improvement > best_improvement:
                best_improvement = improvement
                best_edge = (i, j)
        if best_edge is None:
            break
        i, j = best_edge
        u = np.zeros(n)
        u[i] = 1
        u[j] = -1
        L_current = L_current + np.outer(u, u)
        # print(L_current)
        # Update pseudoinverse and its square using Sherman-Morrison
        L2_pinv = update_pseudoinverse_square(L_pinv, L2_pinv, u)
        L_pinv = update_pseudoinverse(L_pinv, u)
        added_edges.append(best_edge)
        # print(best_edge)
    diag = np.diag(L_pinv)
    R_new = diag[:, np.newaxis] + diag[np.newaxis, :] - 2 * L_pinv
    return R_new

In [ ]:
c_1 = k_median_one_swap_until_optimal(R, 11)
kmed_resistance_matrix_1 =  greedy_add_edges_kmedian(laplacian, 5,c_1)
kirchoff_kmed_1 = kmed_resistance_matrix_1.sum()/2
print(kirchoff_kmed_1/math.comb(n_nodes,2))

In [ ]:
c_2 = k_median_one_swap_until_optimal(R, 21)
kmed_resistance_matrix_2 =  greedy_add_edges_kmedian(laplacian, 10,c_2)
kirchoff_kmed_2 = kmed_resistance_matrix_2.sum()/2
print(kirchoff_kmed_2/math.comb(n_nodes,2))

In [ ]:
c_3 = k_median_one_swap_until_optimal(R, 31)
kmed_resistance_matrix_3 =  greedy_add_edges_kmedian(laplacian, 15,c_3)
kirchoff_kmed_3 = kmed_resistance_matrix_3.sum()/2
print(kirchoff_kmed_3/math.comb(n_nodes,2))

In [ ]:
c_4 = k_median_one_swap_until_optimal(R, 41)
kmed_resistance_matrix_4 =  greedy_add_edges_kmedian(laplacian, 20,c_4)
kirchoff_kmed_4 = kmed_resistance_matrix_4.sum()/2
print(kirchoff_kmed_4/math.comb(n_nodes,2))

In [ ]:
c_5 = k_median_one_swap_until_optimal(R, 51)
kmed_resistance_matrix_5 =  greedy_add_edges_kmedian(laplacian, 25,c_5)
kirchoff_kmed_5 = kmed_resistance_matrix_5.sum()/2
print(kirchoff_kmed_5/math.comb(n_nodes,2))